# Notebook 09: Multi-Agent Orchestration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/09-multi-agent.ipynb)

## Learning Objectives

By the end of this notebook you will be able to:

1. Define specialized agent classes with role, tools, and memory
2. Build an orchestrator that decomposes tasks and delegates to agents
3. Implement agent-to-agent communication protocols
4. Decompose complex tasks into parallel subtasks
5. Execute multiple agents concurrently and synthesize results
6. Handle conflicts and inconsistencies between agent outputs

---
## 1. Setup and Installation

In [ ]:
!pip install openai --quiet

---
## 2. Configuration

In [ ]:
import os
import json
import time
import random
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Mock responses will be used.")
    print("Set it with: os.environ['OPENAI_API_KEY'] = 'your-key-here'")

client = OpenAI(api_key=api_key or "sk-mock-key")
MODEL = "gpt-4o-mini"

---
## 3. Agent Class Definition (Role, Tools, Memory)

Each agent has a role-specific system prompt, a list of capabilities,
and conversational memory that persists across interactions.

In [ ]:
class AgentRole(str, Enum):
    RESEARCHER = "researcher"
    WRITER     = "writer"
    REVIEWER   = "reviewer"


@dataclass
class Message:
    """A message passed between agents."""
    sender: str
    recipient: str
    content: str
    msg_type: str = "task"       # task | result | feedback | final
    metadata: dict = field(default_factory=dict)


@dataclass
class AgentConfig:
    """Configuration for an agent."""
    name: str
    role: AgentRole
    system_prompt: str
    capabilities: list[str]


AGENT_CONFIGS = {
    AgentRole.RESEARCHER: AgentConfig(
        name="Researcher",
        role=AgentRole.RESEARCHER,
        system_prompt=(
            "You are a research specialist. Gather information, analyze "
            "requirements, identify technologies, and produce structured "
            "research summaries with trade-off analysis."
        ),
        capabilities=["information_gathering", "technology_analysis",
                      "requirements_analysis"],
    ),
    AgentRole.WRITER: AgentConfig(
        name="Writer",
        role=AgentRole.WRITER,
        system_prompt=(
            "You are a technical writer. Produce clear, well-structured "
            "content based on research findings. Include code examples, "
            "diagrams in text form, and actionable recommendations."
        ),
        capabilities=["content_creation", "technical_writing",
                      "code_examples"],
    ),
    AgentRole.REVIEWER: AgentConfig(
        name="Reviewer",
        role=AgentRole.REVIEWER,
        system_prompt=(
            "You are a senior technical reviewer. Review content for "
            "accuracy, completeness, clarity, and best practices. Provide "
            "specific, actionable feedback with severity levels."
        ),
        capabilities=["review", "fact_checking", "quality_assurance"],
    ),
}

print("Agent roles defined:")
for role, cfg in AGENT_CONFIGS.items():
    print(f"  {cfg.name} ({role.value}): {', '.join(cfg.capabilities)}")

In [ ]:
class Agent:
    """An individual agent with LLM reasoning and memory."""

    # Pre-built mock responses keyed by role
    MOCK_RESPONSES = {
        AgentRole.RESEARCHER: (
            "## Research Findings\n\n"
            "### Key Technologies\n"
            "- **RAG (Retrieval-Augmented Generation)**: Best for grounding LLM "
            "responses in factual data. Reduces hallucinations by 40-60%.\n"
            "- **Vector databases**: Pinecone, Weaviate, and Chroma are leading options.\n"
            "- **Embedding models**: OpenAI text-embedding-3-small offers good "
            "cost/performance balance.\n\n"
            "### Trade-offs\n"
            "- RAG adds latency (200-500ms per retrieval step)\n"
            "- Fine-tuning is better for style/format; RAG is better for facts\n"
            "- Hybrid approaches (RAG + fine-tuning) yield best results"
        ),
        AgentRole.WRITER: (
            "## Technical Guide: Building a RAG-Powered Q&A System\n\n"
            "### Architecture Overview\n"
            "The system has three main components:\n"
            "1. **Document Ingestion** -- chunk documents, generate embeddings, "
            "store in vector DB\n"
            "2. **Retrieval** -- embed user query, find top-k similar chunks\n"
            "3. **Generation** -- pass retrieved context + query to LLM\n\n"
            "### Example Code\n"
            "```python\n"
            "def answer_question(query: str, top_k: int = 3) -> str:\n"
            "    # 1. Embed the query\n"
            "    query_vec = embed(query)\n"
            "    # 2. Retrieve relevant chunks\n"
            "    chunks = vector_db.search(query_vec, top_k=top_k)\n"
            "    # 3. Generate answer with context\n"
            "    context = '\\n'.join(c.text for c in chunks)\n"
            "    return llm.generate(f'Context: {context}\\nQ: {query}')\n"
            "```\n\n"
            "### Recommendations\n"
            "- Start with chunk sizes of 500-1000 tokens\n"
            "- Use overlap of 10-20% between chunks\n"
            "- Monitor retrieval precision as your primary metric"
        ),
        AgentRole.REVIEWER: (
            "## Review Feedback\n\n"
            "### Overall Assessment: APPROVE with minor revisions\n\n"
            "### Findings\n"
            "1. **[MINOR]** Add error handling for the embed() and vector_db.search() "
            "calls -- both can fail in production\n"
            "2. **[MINOR]** Specify recommended embedding model explicitly\n"
            "3. **[MAJOR]** Missing section on evaluation -- how to measure "
            "retrieval quality (MRR, recall@k) and generation quality (faithfulness)\n"
            "4. **[MINOR]** Add a note about document freshness and re-indexing\n"
            "5. **[MINOR]** Consider adding a caching layer for repeated queries\n\n"
            "### Accuracy Check\n"
            "- The 40-60% hallucination reduction claim is consistent with published "
            "benchmarks\n"
            "- Chunk size recommendations are reasonable and well-sourced"
        ),
    }

    def __init__(self, config: AgentConfig):
        self.config = config
        self.memory: list[Message] = []

    def process(self, message: Message) -> Message:
        """Process an incoming message and produce a response."""
        self.memory.append(message)

        # Build conversation for the LLM
        llm_messages = [
            {"role": "system", "content": self.config.system_prompt},
            {"role": "user", "content": message.content},
        ]

        try:
            response = client.chat.completions.create(
                model=MODEL, messages=llm_messages, temperature=0.7
            )
            content = response.choices[0].message.content
        except Exception as e:
            print(f"API call failed: {e}")
            content = self.MOCK_RESPONSES.get(
                self.config.role,
                f"Mock response from {self.config.name}."
            )
            print("Using mock response for demonstration.")

        reply = Message(
            sender=self.config.name, recipient=message.sender,
            content=content, msg_type="result"
        )
        self.memory.append(reply)
        return reply


# Quick test
researcher = Agent(AGENT_CONFIGS[AgentRole.RESEARCHER])
test_msg = Message(sender="orchestrator", recipient="Researcher",
                   content="Research how to build a RAG system.")
result = researcher.process(test_msg)
print(f"{result.sender} responded ({len(result.content)} chars):")
print(result.content[:300])

---
## 4. Orchestrator Pattern

The orchestrator manages the multi-agent workflow: it decomposes the user
request into subtasks, assigns them to agents, manages dependencies,
and synthesizes the final result.

In [ ]:
@dataclass
class TaskStep:
    """One step in the task plan."""
    agent_role: AgentRole
    description: str
    depends_on: list[int] = field(default_factory=list)


class Orchestrator:
    """Multi-agent orchestrator with task decomposition."""

    def __init__(self, max_iterations: int = 10):
        self.agents: dict[AgentRole, Agent] = {}
        self.message_log: list[Message] = []
        self.max_iterations = max_iterations

        # Create agents
        for role, config in AGENT_CONFIGS.items():
            self.agents[role] = Agent(config)

    def decompose(self, request: str) -> list[TaskStep]:
        """Decompose a request into ordered subtasks."""
        return [
            TaskStep(AgentRole.RESEARCHER,
                     f"Research the best approaches for: {request}",
                     depends_on=[]),
            TaskStep(AgentRole.WRITER,
                     "Based on the research, write a comprehensive guide.",
                     depends_on=[0]),
            TaskStep(AgentRole.REVIEWER,
                     "Review the guide for accuracy, completeness, and clarity.",
                     depends_on=[1]),
        ]

    def execute(self, request: str) -> dict:
        """Execute the full multi-agent workflow."""
        plan = self.decompose(request)
        results: dict[int, Message] = {}

        print(f"Orchestrator: {len(plan)} steps planned")
        print("=" * 60)

        for idx, step in enumerate(plan):
            agent = self.agents[step.agent_role]

            # Build context from dependencies
            context = step.description
            for dep in step.depends_on:
                if dep in results:
                    context += (f"\n\n--- Output from "
                                f"{results[dep].sender} ---\n"
                                f"{results[dep].content}")

            print(f"\n--- Step {idx + 1}: {agent.config.name} ---")
            print(f"  Task: {step.description[:80]}")

            msg = Message(sender="orchestrator",
                          recipient=agent.config.name,
                          content=context, msg_type="task")
            self.message_log.append(msg)

            reply = agent.process(msg)
            self.message_log.append(reply)
            results[idx] = reply

            preview = reply.content[:120].replace('\n', ' ')
            print(f"  Result: {preview}...")

        return {
            "request": request,
            "steps": len(plan),
            "sections": [
                {"agent": plan[i].agent_role.value,
                 "output": results[i].content}
                for i in sorted(results)
            ],
            "total_messages": len(self.message_log),
        }


print("Orchestrator class defined.")

---
## 5. Agent Communication Protocol

A `MessageBus` provides structured message routing between agents,
with full logging for debugging and auditability.

In [ ]:
class MessageBus:
    """Central message bus for inter-agent communication."""

    def __init__(self):
        self.log: list[Message] = []
        self.agents: dict[str, Agent] = {}

    def register(self, agent: Agent):
        self.agents[agent.config.name] = agent

    def send(self, message: Message) -> Optional[Message]:
        self.log.append(message)
        agent = self.agents.get(message.recipient)
        if agent is None:
            err = Message(sender="system", recipient=message.sender,
                          content=f"Unknown agent: {message.recipient}",
                          msg_type="error")
            self.log.append(err)
            return err
        reply = agent.process(message)
        self.log.append(reply)
        return reply

    def trace(self) -> str:
        lines = []
        for i, m in enumerate(self.log):
            preview = m.content[:80].replace('\n', ' ')
            lines.append(f"[{i+1}] {m.sender} -> {m.recipient} "
                         f"[{m.msg_type}]: {preview}")
        return "\n".join(lines)


# Test message passing
bus = MessageBus()
for role, cfg in AGENT_CONFIGS.items():
    bus.register(Agent(cfg))

msg = Message(sender="orchestrator", recipient="Researcher",
              content="Research best practices for prompt engineering.")
reply = bus.send(msg)
print(f"Reply from {reply.sender} ({len(reply.content)} chars)")
print(f"\nMessage trace:\n{bus.trace()}")

---
## 6. Task Decomposition

For complex requests, the orchestrator uses the LLM (or rules) to break
the task into subtasks with dependency ordering.

In [ ]:
def smart_decompose(request: str) -> list[TaskStep]:
    """Use the LLM to decompose a task into agent subtasks."""
    prompt = (
        "Decompose this task into 3 subtasks for a Researcher, Writer, "
        "and Reviewer team. Return JSON array:\n"
        '[{"role": "researcher|writer|reviewer", "task": str, '
        '"depends_on": [int]}]\n\n'
        f"Task: {request}\n\nReturn ONLY valid JSON."
    )
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        raw = response.choices[0].message.content
        import re
        m = re.search(r'\[.*\]', raw, re.DOTALL)
        steps_data = json.loads(m.group() if m else raw)
    except Exception as e:
        print(f"API call failed: {e}")
        print("Using mock response for demonstration.")
        steps_data = [
            {"role": "researcher",
             "task": f"Research current best practices for: {request}",
             "depends_on": []},
            {"role": "writer",
             "task": "Write a comprehensive technical guide based on research.",
             "depends_on": [0]},
            {"role": "reviewer",
             "task": "Review for accuracy, completeness, and actionability.",
             "depends_on": [1]},
        ]

    role_map = {"researcher": AgentRole.RESEARCHER,
                "writer": AgentRole.WRITER,
                "reviewer": AgentRole.REVIEWER}
    steps = []
    for s in steps_data:
        role = role_map.get(s["role"], AgentRole.RESEARCHER)
        steps.append(TaskStep(role, s["task"], s.get("depends_on", [])))
    return steps


plan = smart_decompose("Create a guide for deploying LLMs to production")
print("Task Plan:")
for i, step in enumerate(plan):
    deps = f" (after step {step.depends_on})" if step.depends_on else ""
    print(f"  Step {i}: [{step.agent_role.value}] {step.description}{deps}")

---
## 7. Parallel Agent Execution

When subtasks have no dependencies on each other, they can run in parallel
using threads.

In [ ]:
def execute_parallel(agents: dict[AgentRole, Agent],
                     tasks: dict[str, str]) -> dict[str, Message]:
    """Run independent agent tasks in parallel."""
    results = {}

    def _run(role_str, task_text):
        role = AgentRole(role_str)
        agent = agents[role]
        msg = Message(sender="orchestrator",
                      recipient=agent.config.name,
                      content=task_text, msg_type="task")
        return role_str, agent.process(msg)

    with ThreadPoolExecutor(max_workers=3) as pool:
        futures = {pool.submit(_run, r, t): r for r, t in tasks.items()}
        for future in as_completed(futures):
            role_str, reply = future.result()
            results[role_str] = reply
            print(f"  Completed: {reply.sender} ({len(reply.content)} chars)")

    return results


# Demo parallel execution of independent research tasks
agents = {role: Agent(cfg) for role, cfg in AGENT_CONFIGS.items()}

parallel_tasks = {
    "researcher": "Research vector database options for RAG systems.",
    "writer": "Write an introduction to retrieval-augmented generation.",
}

print("Running parallel tasks:")
t0 = time.time()
results = execute_parallel(agents, parallel_tasks)
elapsed = time.time() - t0
print(f"\nCompleted in {elapsed:.2f}s")
for role, reply in results.items():
    print(f"\n  [{role}] {reply.content[:100].replace(chr(10), ' ')}...")

---
## 8. Result Synthesis

After all agents complete their tasks, the orchestrator synthesizes their
outputs into a coherent final deliverable.

In [ ]:
def synthesize_results(sections: list[dict]) -> str:
    """Synthesize agent outputs into a final deliverable."""
    prompt = (
        "You are an editor. Combine the following sections from different "
        "specialists into a single coherent document. Preserve key information "
        "and maintain a consistent tone.\n\n"
    )
    for sec in sections:
        prompt += f"--- {sec['agent'].upper()} OUTPUT ---\n{sec['output']}\n\n"
    prompt += "Produce the final combined document."

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5
        )
        result = response.choices[0].message.content
    except Exception as e:
        print(f"API call failed: {e}")
        print("Using mock response for demonstration.")
        result = (
            "# Building a RAG-Powered Q&A System\n\n"
            "## Executive Summary\n"
            "This guide covers how to build a production-ready RAG system, "
            "combining research findings, implementation guidance, and "
            "quality review feedback.\n\n"
            "## Key Technologies\n"
            "RAG reduces hallucinations by 40-60%. Recommended stack: "
            "OpenAI embeddings + Pinecone/Chroma vector store.\n\n"
            "## Implementation\n"
            "1. Chunk documents (500-1000 tokens, 10-20% overlap)\n"
            "2. Embed and store in vector DB\n"
            "3. At query time: embed query, retrieve top-k, generate answer\n\n"
            "## Quality Notes\n"
            "- Add error handling for all external calls\n"
            "- Include evaluation metrics (MRR, recall@k, faithfulness)\n"
            "- Consider caching for repeated queries"
        )

    print(result)
    return result


# Run full workflow and synthesize
orch = Orchestrator()
workflow = orch.execute("Build a RAG-powered Q&A system for technical documentation")

print(f"\n{'='*60}")
print("SYNTHESIZED FINAL OUTPUT")
print("=" * 60)
final = synthesize_results(workflow["sections"])

---
## 9. Conflict Resolution

When agents disagree, the orchestrator must resolve conflicts. This can
involve re-querying agents, applying voting, or escalating to a
"judge" agent.

In [ ]:
class ConflictResolver:
    """Resolve disagreements between agent outputs."""

    def detect_conflicts(self, outputs: dict[str, str]) -> list[dict]:
        """Detect potential conflicts between agent outputs."""
        prompt = (
            "Analyze these outputs from different agents and identify "
            "any contradictions or inconsistencies. Return JSON:\n"
            '{"conflicts": [{"description": str, "agents": [str], '
            '"severity": "low|medium|high"}], "has_conflicts": bool}\n\n'
        )
        for name, content in outputs.items():
            prompt += f"--- {name} ---\n{content[:500]}\n\n"
        prompt += "Return ONLY valid JSON."

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            raw = response.choices[0].message.content
            import re
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            result = json.loads(m.group() if m else raw)
        except Exception as e:
            print(f"API call failed: {e}")
            print("Using mock response for demonstration.")
            result = {
                "has_conflicts": True,
                "conflicts": [
                    {"description": "Researcher recommends Pinecone but Writer "
                                    "uses Chroma in examples",
                     "agents": ["Researcher", "Writer"],
                     "severity": "medium"},
                    {"description": "Reviewer notes missing evaluation section "
                                    "that Writer should have included",
                     "agents": ["Writer", "Reviewer"],
                     "severity": "high"},
                ]
            }
        return result

    def resolve(self, conflict: dict, agent_outputs: dict[str, str]) -> str:
        """Resolve a specific conflict using a judge approach."""
        prompt = (
            f"Conflict: {conflict['description']}\n"
            f"Severity: {conflict['severity']}\n"
            f"Agents involved: {conflict['agents']}\n\n"
            "As a senior judge, provide a resolution that:\n"
            "1. Identifies which agent's position is more correct\n"
            "2. Provides a merged recommendation\n"
            "Be concise (2-3 sentences)."
        )
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            result = response.choices[0].message.content
        except Exception as e:
            print(f"API call failed: {e}")
            result = ("Mock: Both vector databases are valid. Use Chroma for "
                      "development/prototyping and Pinecone for production. "
                      "Update examples to show both options.")
            print("Using mock response for demonstration.")
        return result


# Demo conflict resolution
resolver = ConflictResolver()

agent_outputs = {
    "Researcher": "Recommend Pinecone for production vector storage.",
    "Writer": "Example code uses Chroma as the vector database.",
    "Reviewer": "Missing evaluation metrics section.",
}

conflicts = resolver.detect_conflicts(agent_outputs)
print(f"Conflicts found: {conflicts['has_conflicts']}")
for c in conflicts.get("conflicts", []):
    print(f"\n  [{c['severity'].upper()}] {c['description']}")
    resolution = resolver.resolve(c, agent_outputs)
    print(f"  Resolution: {resolution}")

---
## 10. Full Multi-Agent Run with Trace

Execute a complete multi-agent workflow and inspect the full communication
trace for debugging.

In [ ]:
# Fresh orchestrator run
orch2 = Orchestrator()
result = orch2.execute(
    "Write a best-practices guide for prompt engineering in production systems"
)

print(f"\n{'='*60}")
print("EXECUTION SUMMARY")
print("=" * 60)
print(f"Steps completed: {result['steps']}")
print(f"Total messages:  {result['total_messages']}")

print(f"\nAgent outputs:")
for sec in result["sections"]:
    lines = sec["output"].split("\n")
    print(f"\n  [{sec['agent']}] ({len(sec['output'])} chars, {len(lines)} lines)")
    print(f"  First line: {lines[0][:80]}")

print(f"\nMessage trace:")
for i, msg in enumerate(orch2.message_log):
    direction = f"{msg.sender} -> {msg.recipient}"
    print(f"  [{i+1}] {direction:35s} [{msg.msg_type}] "
          f"{msg.content[:50].replace(chr(10), ' ')}...")

---
## Summary

This notebook built a complete multi-agent orchestration system:

1. **Agent Class** -- Specialized agents (Researcher, Writer, Reviewer) with role-specific prompts, capabilities, and memory.
2. **Orchestrator Pattern** -- Task decomposition, dependency management, and sequential delegation.
3. **Communication Protocol** -- MessageBus for structured inter-agent messaging with full trace logging.
4. **Task Decomposition** -- LLM-powered breakdown of complex requests into agent subtasks.
5. **Parallel Execution** -- Concurrent agent execution for independent subtasks using thread pools.
6. **Result Synthesis** -- Combining multiple agent outputs into a coherent final deliverable.
7. **Conflict Resolution** -- Detecting and resolving contradictions between agent outputs.

### Key Takeaways

- Each agent should have a clear, narrow responsibility.
- The orchestrator is the critical component -- it determines task flow and quality.
- Always log all inter-agent messages for debugging and evaluation.
- Use parallel execution for independent subtasks to reduce latency.
- Plan for conflicts -- agents may produce inconsistent outputs that need resolution.